In [24]:
import sys
import numpy as np
import pandas as pd

## Inputs 

In [50]:
No_of_minterms = int(input("Enter the number of minterms: "))
literals_count = int(input("enter number of literals: "))
minterms = []#[0, 2, 5, 6, 7, 8, 10, 13, 14, 15]
x = 0
for i in range(No_of_minterms):
    x = int(input(f"Enter minterm {i+1}: "))
    minterms.append(x if x < 2**literals_count else sys.exit("Invalid minterm"))

print(minterms)

Enter the number of minterms:  10
enter number of literals:  4
Enter minterm 1:  0
Enter minterm 2:  2
Enter minterm 3:  5
Enter minterm 4:  6
Enter minterm 5:  7
Enter minterm 6:  8
Enter minterm 7:  10
Enter minterm 8:  13
Enter minterm 9:  14
Enter minterm 10:  15


[0, 2, 5, 6, 7, 8, 10, 13, 14, 15]


## Making the groups

In [51]:
def No_of_ones(binary_str):
    return binary_str.count('1')

In [66]:
groups_d = {}
for i in range(literals_count + 1):
    groups_d[i] = []

for x in minterms:
    binary_str = f"{x:0{literals_count}b}" 
    print(binary_str)
    ones_count = No_of_ones(binary_str)
    # Append the tuple: (binary_string, set_of_minterms)
    groups_d[ones_count].append((binary_str, {x}))
print(groups_d)

0000
0010
0101
0110
0111
1000
1010
1101
1110
1111
{0: [('0000', {0})], 1: [('0010', {2}), ('1000', {8})], 2: [('0101', {5}), ('0110', {6}), ('1010', {10})], 3: [('0111', {7}), ('1101', {13}), ('1110', {14})], 4: [('1111', {15})]}


## Combining minterms from adjacent groups $\Rightarrow$ Prime Implicants

##### Funtion to check single bit change 

In [54]:
import numpy as np

def single_bit(x, y, edge_case = False):
    merged_list = []
    count_y = np.zeros(len(y))
    count = 0
    for i in range(len(x)):
        bin1 = x[i][0]
        min1 = x[i][1]
        count_x = 0 
        for j in range(len(y)):
            bin2 = y[j][0]
            min2 = y[j][1]
            
            diff_count = 0
            temp_bin = ""
            is_illegal_match = False
            
            for k in range(len(bin1)):
                if (bin1[k] == '-' and bin2[k] != '-') or (bin2[k] == '-' and bin1[k] != '-'):
                    is_illegal_match = True
                    break
                
                if bin1[k] != bin2[k]:
                    diff_count += 1
                    temp_bin += '-' 
                else:
                    temp_bin += bin1[k]
            
            if is_illegal_match:
                continue
                
            if diff_count == 1:
                count = count + 1
                temp_set = min1 | min2 
                temp_tuple = (temp_bin, temp_set)
                
                if temp_tuple not in merged_list:
                    merged_list.append(temp_tuple)
                    count_x = count_x + 1
                    count_y[j] = count_y[j] + 1
        if count_x == 0: 
            merged_list.append(x[i])        
        else: continue
    if edge_case == True:
            for j in range(len(y)):
                if count_y[j] == 0: 
                    merged_list.append(y[j])
    return merged_list, count


prime_implicants = []
round_dashes = 1 

while True:
    x = len(groups_d)
    total_round_merges = 0 
    
    if x < 2:
        for key in groups_d:
            for u in groups_d[key]:
                if u not in prime_implicants:
                    prime_implicants.append(u)
        break
        
    for i in range(x - 1):
        is_last_pairing = (i == x - 2)
        
        new_combined_group, match_count = single_bit(groups_d[i], groups_d[i+1], is_last_pairing)
        total_round_merges += match_count
        
        if is_last_pairing:
            groups_d[i+1] = []
        groups_d[i] = []
        
        for u in new_combined_group:
            # Dash check cleanly separates our winners from our leftovers
            if u[0].count('-') == round_dashes:
                groups_d[No_of_ones(u[0])].append(u)
            else:
                if u not in prime_implicants:
                    prime_implicants.append(u)
                    
    if total_round_merges == 0:
        for key in groups_d:
            for u in groups_d[key]:
                if u not in prime_implicants:
                    prime_implicants.append(u)
        break
        
    round_dashes += 1

## Prime Implicants Chart to find out the essential prime implicants
prime_implicants is basically a list of tuples

In [60]:
# 1. Clean out the "sub-term ghost duplicates" from Phase 1 first
final_clean_pis = [
    pi for pi in prime_implicants 
    if not any(pi[1].issubset(other[1]) and pi[1] != other[1] for other in prime_implicants)
]

# 2. Extract clean string row labels
pi_rows = [str(pi[0]) for pi in final_clean_pis]

# 3. Initialize the empty False DataFrame
df = pd.DataFrame(0, index=pi_rows, columns=minterms)

# 4. Safely populate it row-by-row
for i in range(len(final_clean_pis)):
    binary_string = str(final_clean_pis[i][0])
    
    # Force convert the minterm set to a pure list of standard integers
    covered_minterms = [int(m) for m in final_clean_pis[i][1]]
    
    # Set the value to 1 for the intersections
    df.loc[binary_string, covered_minterms] = 1

print(df)

      0   2   5   6   7   8   10  13  14  15
-0-0   1   1   0   0   0   1   1   0   0   0
--10   0   1   0   1   0   0   1   0   1   0
-1-1   0   0   1   0   1   0   0   1   0   1
-11-   0   0   0   1   1   0   0   0   1   1


In [61]:
essential_prime_implicants = set()
for m in minterms:
    if np.sum(df[m]) == 1 :
        corresponding_pi = df.index[df[m] == 1][0]
        essential_prime_implicants = essential_prime_implicants|{corresponding_pi}#that corresponding one's prime implicant
essential_prime_implicants = list(essential_prime_implicants)
print(essential_prime_implicants)

['-0-0', '-1-1']


In [62]:
covered_minterms = set()

for epi in essential_prime_implicants:
    # Get the columns (minterms) where this EPI row is True
    matched_columns = df.columns[df.loc[epi] == True]
    # Add them to our covered set
    covered_minterms.update(matched_columns)

# Convert to a list so pandas can use it to drop columns
covered_minterms = list(covered_minterms)
# 1. Drop the columns of minterms we already covered
reduced_df = df.drop(columns=covered_minterms)

# 2. Drop the rows of the EPIs we already selected
reduced_df = reduced_df.drop(index=essential_prime_implicants)
print(reduced_df)

      6   14
--10   1   1
-11-   1   1


In [64]:
temp_df = reduced_df.copy()
result = [] 

# In pandas, checking if a dataframe is empty is done using .empty
while not temp_df.empty:
    max_score = -1
    max_score_pi = ""
    le = len(temp_df.index)
    
    for i in range(le):
        row_sum = np.sum(temp_df.iloc[i]) 
        if row_sum > max_score:
            max_score = row_sum
            max_score_pi = temp_df.index[i] 
            
    # Append our winning row to our results list
    result.append(max_score_pi)
    
    matched_columns_temp = temp_df.columns[temp_df.loc[max_score_pi] == True]
    
    covered_minterms_temp = list(matched_columns_temp)
    
    temp_df = temp_df.drop(columns=covered_minterms_temp)
    temp_df = temp_df.drop(index=max_score_pi)
result = result+essential_prime_implicants
print(result)

['--10', '-0-0', '-1-1']


In [65]:
def print_final_expression(strings, variables=['A', 'B', 'C', 'D']):
    final_terms = []
    for string in strings:
        term = ""
        for i in range(len(string)):
            if string[i] == '1':
                term += variables[i]
            elif string[i] == '0':
                term += variables[i] + "'"  # ' for NOT
        final_terms.append(term)
    return " + ".join(final_terms)

# Print your ultimate solution!
print("Your Minimized Equation:", print_final_expression(['--10', '-0-0', '-1-1']))

Your Minimized Equation: CD' + B'D' + BD
